# Customer Segmentation using Clustering

**Topic:** Unsupervised Learning - K-Means Clustering  
**Objective:** Segment mall customers using Age, Annual Income, and Spending Score, visualize customer groups, and label the groups as Low Spenders, Medium Spenders, and High Spenders.

Dataset: Mall Customers Dataset from Kaggle via `kagglehub`.

> Run the cells from top to bottom. The first run may require internet access and Kaggle authentication, depending on your environment.


## 1. Install required libraries
If needed, run the following in a notebook cell. In Google Colab/Jupyter, remove the `#` at the start to install packages.


In [ ]:
# %pip install kagglehub[pandas-datasets] pandas scikit-learn matplotlib seaborn


## 2. Load the dataset
The commonly used Kaggle Mall Customers dataset contains `Mall_Customers.csv`. If loading fails, check the dataset files and update `file_path`.


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from kagglehub import KaggleDatasetAdapter
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

file_path = "Mall_Customers.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "vjchoudhary7/customer-segmentation-tutorial-in-python",
    file_path,
)

print("First five records:")
display(df.head())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


## 3. Inspect and prepare the data
K-Means will use Age, Annual Income, and Spending Score. Customer ID is not used because it is only an identifier.


In [ ]:
df = df.copy()
df.columns = [column.strip() for column in df.columns]

expected_features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
missing_features = [column for column in expected_features if column not in df.columns]

if missing_features:
    raise ValueError(
        f"Missing expected columns: {missing_features}. Available columns: {df.columns.tolist()}. "
        "Check the dataset file and update expected_features if necessary."
    )

features = expected_features
X = df[features].copy()

print("Features used for clustering:", features)
print("Missing values in selected features:")
display(X.isna().sum())
display(X.describe())


## 4. Visualize the original customer data
These plots show the relationship between Age, Annual Income, and Spending Score before clustering.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(data=df, x="Age", y="Spending Score (1-100)", ax=axes[0])
axes[0].set_title("Age vs Spending Score")

sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)", ax=axes[1])
axes[1].set_title("Income vs Spending Score")

sns.scatterplot(data=df, x="Age", y="Annual Income (k$)", ax=axes[2])
axes[2].set_title("Age vs Annual Income")

plt.tight_layout()
plt.show()


## 5. Scale the features
K-Means is distance-based, so scaling keeps Age, Income, and Spending Score on comparable ranges.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(X_scaled, columns=features)
display(scaled_df.head())


## 6. Find a useful number of clusters
The elbow method compares K-Means inertia for different values of `k`. The project labels customers into three spender groups, so the final model below uses `k=3`.


In [ ]:
inertia_values = []
k_values = range(1, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertia_values, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()


## 7. Apply K-Means clustering
The assignment asks for customer groups labeled as Low, Medium, and High Spenders, so `k=3` is used.


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df["Cluster"] = clusters

print("Cluster counts:")
display(df["Cluster"].value_counts().sort_index())
print(f"K-Means inertia: {kmeans.inertia_:.2f}")
print(f"Silhouette score: {silhouette_score(X_scaled, clusters):.4f}")


## 8. Label customer groups
Cluster numbers are assigned automatically by K-Means. To create business-friendly labels, clusters are sorted by their average Spending Score.


In [ ]:
cluster_summary = df.groupby("Cluster")[features].mean().round(2)
cluster_order = cluster_summary.sort_values("Spending Score (1-100)").index.tolist()

spender_labels = {
    cluster_order[0]: "Low Spenders",
    cluster_order[1]: "Medium Spenders",
    cluster_order[2]: "High Spenders",
}

df["Customer_Group"] = df["Cluster"].map(spender_labels)

print("Cluster summary with business labels:")
display(
    df.groupby(["Cluster", "Customer_Group"])[features]
    .mean()
    .round(2)
)


## 9. Visualize customer groups
These scatter plots show how the customer segments differ by Age, Annual Income, and Spending Score.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
palette = "Set2"

sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Customer_Group",
    palette=palette,
    s=80,
    edgecolor="black",
    ax=axes[0],
)
axes[0].set_title("Income vs Spending Score")

sns.scatterplot(
    data=df,
    x="Age",
    y="Spending Score (1-100)",
    hue="Customer_Group",
    palette=palette,
    s=80,
    edgecolor="black",
    ax=axes[1],
)
axes[1].set_title("Age vs Spending Score")

sns.scatterplot(
    data=df,
    x="Age",
    y="Annual Income (k$)",
    hue="Customer_Group",
    palette=palette,
    s=80,
    edgecolor="black",
    ax=axes[2],
)
axes[2].set_title("Age vs Annual Income")

for axis in axes:
    axis.legend(title="Customer Group")

plt.tight_layout()
plt.show()


In [ ]:
fig = plt.figure(figsize=(10, 7))
axis = fig.add_subplot(111, projection="3d")

groups = df["Customer_Group"].unique()
colors = sns.color_palette("Set2", n_colors=len(groups))

for group, color in zip(groups, colors):
    group_data = df[df["Customer_Group"] == group]
    axis.scatter(
        group_data["Age"],
        group_data["Annual Income (k$)"],
        group_data["Spending Score (1-100)"],
        label=group,
        color=color,
        s=60,
    )

axis.set_title("Customer Segments by Age, Income, and Spending Score")
axis.set_xlabel("Age")
axis.set_ylabel("Annual Income (k$)")
axis.set_zlabel("Spending Score (1-100)")
axis.legend(title="Customer Group")
plt.tight_layout()
plt.show()


## 10. Final segmented customer table
This table shows the original customer records with the predicted cluster and business-friendly group label.


In [ ]:
final_columns = [column for column in ["CustomerID", "Gender"] if column in df.columns] + features + ["Cluster", "Customer_Group"]
final_segments = df[final_columns].copy()
display(final_segments.head(20))

print("Number of customers in each group:")
display(final_segments["Customer_Group"].value_counts())


## Conclusion
K-Means clustering was used to segment mall customers based on Age, Annual Income, and Spending Score. The groups were labeled as Low Spenders, Medium Spenders, and High Spenders based on each cluster's average Spending Score. The visualizations show how customer groups differ across spending behavior, income, and age.

**Business outcome:** This segmentation can help a business target promotions, loyalty programs, and marketing campaigns toward different customer groups.

**Submission note:** Run every notebook cell and save the notebook with the output tables and graphs visible before submitting.
